# Arena Sagaz - Treinamento da Rede Neural (Edge AI)

Este notebook processa os dados gerados pelo Minimax e treina a Rede Neural que jogará o Arena Sagaz no app Mobile.

In [1]:
import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Reduz logs internos do TensorFlow para deixar a saída do notebook mais limpa.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
sns.set_theme(style="whitegrid")

# Sementes fixas para reprodutibilidade dos experimentos.
np.random.seed(42)
tf.keras.utils.set_random_seed(42)

print("TensorFlow:", tf.__version__)
print("GPU disponível:", len(tf.config.list_physical_devices('GPU')) > 0)

TensorFlow: 2.10.0
GPU disponível: True


## 1. Carregamento, Tratamento e Split em 2 Etapas

Aqui fazemos:

1. **Leitura** dos lotes `.npz` gerados pelo Minimax (formato com Q-values).  
   Cada arquivo contém: `estados` (matriz do tabuleiro), `rotulos` (argmax string),
   `scores` (vetor 31 × float32 com o Q-value do Minimax para cada traço possível)
   e `labels_canonicos` (ordem determinística dos 31 slots).

2. **Normalização**: pontos fixos (8) → 0 para não estourar gradientes.

3. **Split em duas etapas** (estilo aula):
   - `train_test_split` separa Treino (70%) de um bloco temporário (30%).
   - `train_test_split` divide o bloco temporário em Validação (15%) e Teste (15%).
   - O vetor `scores` é dividido junto com `estados` e `rotulos`.

4. **Permutações de simetria para os scores**: antes de augmentar, pré-calculamos
   3 vetores de permutação de índices (flip-H, flip-V, rot-180°). Quando espelhamos
   o tabuleiro, o score de `H_0_1` deve ir para o slot de `H_0_5` — a permutação
   faz isso com uma indexação `scores[:, perm]` em vez de remapear strings.

5. **Augmentação por simetria APENAS no treino**: o tabuleiro tem 4 simetrias do
   grupo D₂ (retângulo). Cada estado vira 4 exemplos com estado espelhado e vetor
   de scores permutado. Val e Teste **não** são augmentados para medir desempenho real.

6. **Soft targets via Softmax mascarado**: os `scores` são convertidos em distribuição
   de probabilidade com temperatura `T=1.0`. Slots indisponíveis (sentinela `-1e9`)
   são zerados antes da normalização. O resultado é o alvo da `KLDivergence`.

In [2]:
# =========================================================================
# 1.1 LEITURA DOS LOTES (formato com Q-values)
# =========================================================================
arquivos_npz = sorted(glob.glob('../dados/*.npz'))
if not arquivos_npz:
    arquivos_npz = sorted(glob.glob('*.npz'))
print(f"Encontrados {len(arquivos_npz)} arquivos de lote.")

lista_estados, lista_rotulos, lista_scores = [], [], []
labels_canonicos = None

for arquivo in arquivos_npz:
    dados = np.load(arquivo, allow_pickle=False)
    if 'scores' not in dados:
        raise ValueError(
            f"{arquivo} não contém o campo 'scores'. "
            "Apague os dados antigos e gere um novo dataset com o gerador atualizado."
        )
    lista_estados.append(dados['estados'])
    lista_rotulos.append(dados['rotulos'].astype(str))
    lista_scores.append(dados['scores'])           # (lote, 31) float32
    if labels_canonicos is None:
        labels_canonicos = dados['labels_canonicos'].astype(str).tolist()

X_raw    = np.concatenate(lista_estados, axis=0)   # (N, 9, 7)  int8
y_str    = np.concatenate(lista_rotulos, axis=0)   # (N,)       str
S_raw    = np.concatenate(lista_scores,  axis=0)   # (N, 31)    float32
num_classes = len(labels_canonicos)
print(f"Total de tabuleiros: {len(X_raw):,}  |  Classes: {num_classes}")

# Mapeamento canônico label ↔ índice (mesma ordem do vetor de scores do gerador)
dicionario_rotulos  = {lab: i for i, lab in enumerate(labels_canonicos)}
indice_para_rotulo  = {i: lab for lab, i in dicionario_rotulos.items()}

# Normalização: pontos fixos (8) → 0 para estabilizar gradientes.
X_raw = X_raw.astype(np.float32)
X_raw[X_raw == 8] = 0.0
X_raw[X_raw == 9] = 1.0  # Normalizar arestas: 9 -> 1

# =========================================================================
# 1.2 SPLIT EM 2 ETAPAS (Treino 70% / Validação 15% / Teste 15%)
# =========================================================================
X_train_raw, X_temp, y_train_str, y_temp_str, S_train, S_temp = train_test_split(
    X_raw, y_str, S_raw, test_size=0.30, random_state=42,
)
X_val_raw, X_test_raw, y_val_str, y_test_str, S_val, S_test = train_test_split(
    X_temp, y_temp_str, S_temp, test_size=0.50, random_state=42,
)
print(f"Treino: {X_train_raw.shape[0]:,}  |  Val: {X_val_raw.shape[0]:,}  |  Teste: {X_test_raw.shape[0]:,}")

# =========================================================================
# 1.3 PERMUTAÇÕES DE SIMETRIA PARA O VETOR DE SCORES
# Quando espelhamos o tabuleiro, o score da jogada H_0_1 deve ir para o slot
# onde está H_0_5 (flip horizontal), e assim por diante.
# perm[i] = j  →  o score que estava em i passa para j na versão espelhada.
# =========================================================================
def remapeia_label(label: str, flip_linha: bool, flip_coluna: bool) -> str:
    tipo, r_str, c_str = label.split('_')
    r, c = int(r_str), int(c_str)
    if flip_linha:
        r = 8 - r
    if flip_coluna:
        c = 6 - c
    return f"{tipo}_{r}_{c}"

def build_perm(labels_list, flip_linha, flip_coluna):
    idx_map = {l: i for i, l in enumerate(labels_list)}
    return np.array([idx_map[remapeia_label(l, flip_linha, flip_coluna)]
                     for l in labels_list], dtype=np.int32)

perm_flip_col = build_perm(labels_canonicos, False, True)   # espelho horizontal
perm_flip_lin = build_perm(labels_canonicos, True,  False)  # espelho vertical
perm_rot180   = build_perm(labels_canonicos, True,  True)   # rotação 180°

# =========================================================================
# 1.4 AUGMENTAÇÃO POR SIMETRIA (APENAS no treino) — estado + scores
# =========================================================================
def aumenta_por_simetria(X, y_labels, scores):
    aug_X, aug_y, aug_s = [X], [y_labels], [scores]
    # Espelho horizontal (flip de colunas)
    aug_X.append(np.flip(X, axis=2))
    aug_y.append(np.array([remapeia_label(l, False, True) for l in y_labels]))
    aug_s.append(scores[:, perm_flip_col])
    # Espelho vertical (flip de linhas)
    aug_X.append(np.flip(X, axis=1))
    aug_y.append(np.array([remapeia_label(l, True, False) for l in y_labels]))
    aug_s.append(scores[:, perm_flip_lin])
    # Rotação 180°
    aug_X.append(np.flip(np.flip(X, axis=1), axis=2))
    aug_y.append(np.array([remapeia_label(l, True, True) for l in y_labels]))
    aug_s.append(scores[:, perm_rot180])
    return (np.concatenate(aug_X), np.concatenate(aug_y), np.concatenate(aug_s))

X_train_raw, y_train_str, S_train = aumenta_por_simetria(X_train_raw, y_train_str, S_train)
print(f"Treino após augmentação 4×: {X_train_raw.shape[0]:,}")

# =========================================================================
# 1.5 SOFT TARGETS: scores do Minimax → distribuição de probabilidade
# Sentinela -1e9 marca slots indisponíveis (traços já ocupados).
# Temperatura T: menor = mais concentrada (quase one-hot); maior = mais suave.
# Com T=1.0 e empate entre 3 jogadas, cada uma recebe ~33%.
# =========================================================================
TEMPERATURA = 1.0

def scores_para_soft_targets(scores_raw: np.ndarray, T: float = TEMPERATURA) -> np.ndarray:
    s = scores_raw.copy().astype(np.float64)
    mascara = (scores_raw > -1e8).astype(np.float64)
    s[mascara == 0] = -np.inf                          # indisponíveis → 0 após exp
    s -= np.nanmax(np.where(mascara, s, -np.inf), axis=1, keepdims=True)  # estabiliza
    exp_s = np.exp(s / T) * mascara
    return (exp_s / exp_s.sum(axis=1, keepdims=True)).astype(np.float32)

y_train = scores_para_soft_targets(S_train)
y_val   = scores_para_soft_targets(S_val)
y_test  = scores_para_soft_targets(S_test)

# Índices do argmax (melhor jogada) — usados apenas nas métricas de avaliação.
y_train_idx = y_train.argmax(axis=1)
y_val_idx   = y_val.argmax(axis=1)
y_test_idx  = y_test.argmax(axis=1)

# Adiciona canal para a CNN: (N, 9, 7) → (N, 9, 7, 1)
X_train = X_train_raw.reshape(-1, 9, 7, 1)
X_val   = X_val_raw.reshape(-1,   9, 7, 1)
X_test  = X_test_raw.reshape(-1,  9, 7, 1)

print("Shapes finais:")
print(f"  X_train: {X_train.shape}  y_train (soft): {y_train.shape}")
print(f"  X_val:   {X_val.shape}    y_val   (soft): {y_val.shape}")
print(f"  X_test:  {X_test.shape}   y_test  (soft): {y_test.shape}")

Encontrados 60 arquivos de lote.
Total de tabuleiros: 300,000  |  Classes: 31
Treino: 210,000  |  Val: 45,000  |  Teste: 45,000
Treino após augmentação 4×: 840,000
Shapes finais:
  X_train: (840000, 9, 7, 1)  y_train (soft): (840000, 31)
  X_val:   (45000, 9, 7, 1)    y_val   (soft): (45000, 31)
  X_test:  (45000, 9, 7, 1)   y_test  (soft): (45000, 31)


## 2. Arquitetura `BoxNet v3` — CNN Centrada em Caixas com Soft Targets

### Histórico de tentativas

| Tentativa              | Dados  | Treino top1 | Val top1 | Val top3 | Val top5 | OMA   | Loss                   | Sintoma / observação principal                                             |
|------------------------|--------|-------------|----------|----------|----------|-------|------------------------|----------------------------------------------------------------------------|
| CNN ingênua            | 50k    | 4%          | 4%       | —        | —        | —     | CategoricalCE          | Colapso: filtro 3×3 mistura semânticas heterogêneas                        |
| MLP densa              | 50k    | 42%         | 28%      | —        | —        | —     | CategoricalCE          | Overfitting severo: decora coordenadas, sem viés geométrico                |
| BoxNet v1              | 50k    | 60%         | 36%      | —        | —        | —     | CategoricalCE          | Overfit clássico: val_loss subindo na época 10                             |
| BoxNet v2              | 50k    | 46%         | 36%      | 70%      | 83%      | —     | CategoricalCE (smooth) | Plateau — ruído de rótulo por sorteio de empates Minimax                   |
| BoxNet v3 — rodada 1   | 55k    | 34.5%       | 34.4%    | 67.0%    | 80.0%    | —     | KL Div (T=1.0)         | Zero overfitting; teto top-1 por empates na abertura                       |
| BoxNet v3 — rodada 2   | 210k   | 35.5%       | 35.6%    | 68.9%    | 83.4%    | —     | KL Div (T=1.0)         | Top-3/top-5 melhoram com volume; teto top-1 mantido                        |
| BoxNet v3 — rodada 3   | 300k   | 33.1%       | 33.2%    | 65.7%    | 81.5%    | **99.0%** | KL Div (T=0.5)    | T=0.5+sample_weight PIOROU — downweight de abertura destruiu top-3=40.6%  |
| **BoxNet v3 — rodada 4** | **300k** | **?**   | **?**    | **?**    | **?**    | **?** | **KL Div (T=1.0)**     | **Reverte T e sample_weight; mantém 120 épocas — esperado top-1 ≥ 37%**   |

### Mudança principal: `CategoricalCrossentropy` → `KL Divergence`

A `CategoricalCrossentropy` com one-hot punia a rede por distribuir probabilidade
entre jogadas Minimax equivalentes (empates sorteados aleatoriamente). A
`KLDivergence` mede a distância entre **distribuições**: o alvo é agora o vetor
completo de Q-values do Minimax normalizado via Softmax. Jogadas de mesmo score
recebem peso igual — o sinal de treino deixa de ser contraditório.

> **Nota sobre a magnitude da loss:** valores de KLD são diferentes de cross-entropy.
> KLD ≈ 0 significa distribuições idênticas; o importante é acompanhar a *tendência*
> (descendo) e o gap treino/val, não o valor absoluto.

> **Métrica principal — Optimal Move Accuracy (OMA):** mede se a predição top-1 pertence
> ao conjunto de jogadas Minimax-ótimas, independente do desempate canônico. OMA = 99%
> significa que a IA joga de forma ótima em 99% dos estados. O top-1 clássico mede um
> problema diferente (acerto do desempate canônico) e é sistematicamente deprimido por
> estados de abertura com muitas jogadas equivalentes.

In [4]:
# =========================================================================
# CAMADA 0 — TRANSFORMAÇÃO GEOMÉTRICA (9,7,1) → (4,3,5)
# Reorganiza a matriz crua em grid centrado em CAIXAS.
# Convenção canônica (vide gerador_dados/tabuleiro.py):
#   linha PAR  + coluna ÍMPAR → traço Horizontal
#   linha ÍMPAR + coluna PAR  → traço Vertical
#   linha ÍMPAR + coluna ÍMPAR → interior da caixa
# =========================================================================
def para_grid_de_caixas(x):
    x = tf.squeeze(x, axis=-1)                # (B, 9, 7)
    topo     = x[:, 0:8:2, 1:7:2]             # (B,4,3) traço H acima
    base     = x[:, 2:9:2, 1:7:2]             # (B,4,3) traço H abaixo
    esquerda = x[:, 1:8:2, 0:6:2]             # (B,4,3) traço V à esquerda
    direita  = x[:, 1:8:2, 2:7:2]             # (B,4,3) traço V à direita
    interior = x[:, 1:8:2, 1:7:2]             # (B,4,3) dono da caixa
    return tf.stack([topo, base, esquerda, direita, interior], axis=-1)  # (B,4,3,5)


def bloco_residual_separavel(x, filtros, l2=2e-4, dropout=0.15):
    """Bloco residual: 2× SeparableConv2D 3×3 + BN + skip connection."""
    atalho = x
    y = layers.SeparableConv2D(
        filtros, (3, 3), padding='same', use_bias=False,
        depthwise_regularizer=regularizers.l2(l2),
        pointwise_regularizer=regularizers.l2(l2),
    )(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation('relu')(y)
    y = layers.SpatialDropout2D(dropout)(y)

    y = layers.SeparableConv2D(
        filtros, (3, 3), padding='same', use_bias=False,
        depthwise_regularizer=regularizers.l2(l2),
        pointwise_regularizer=regularizers.l2(l2),
    )(y)
    y = layers.BatchNormalization()(y)

    if atalho.shape[-1] != filtros:
        atalho = layers.Conv2D(filtros, (1, 1), padding='same', use_bias=False,
                               kernel_regularizer=regularizers.l2(l2))(atalho)
        atalho = layers.BatchNormalization()(atalho)

    out = layers.Add()([y, atalho])
    out = layers.Activation('relu')(out)
    return out


# =========================================================================
# ARQUITETURA BoxNet v3 — idêntica à v2, mas treinada com KL Divergence
# =========================================================================
L2 = 2e-4

inputs = Input(shape=(9, 7, 1), name='tabuleiro_cru')

x = layers.Lambda(para_grid_de_caixas, name='grid_de_caixas')(inputs)

x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False,
                  kernel_regularizer=regularizers.l2(L2))(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)

x = bloco_residual_separavel(x, 32, l2=L2, dropout=0.15)
x = bloco_residual_separavel(x, 48, l2=L2, dropout=0.20)

gap  = layers.GlobalAveragePooling2D()(x)
flat = layers.Flatten()(x)
h    = layers.Concatenate()([gap, flat])
h    = layers.Dense(96, activation='relu', kernel_regularizer=regularizers.l2(L2))(h)
h    = layers.BatchNormalization()(h)
h    = layers.Dropout(0.5)(h)

outputs = layers.Dense(num_classes, activation='softmax',
                       kernel_regularizer=regularizers.l2(L2),
                       name='jogada')(h)

model = models.Model(inputs, outputs, name='BoxNet_v3_ArenaSagaz')

# KLDivergence: mede distância entre a distribuição prevista e o soft target.
# Não usar label_smoothing — os soft targets já codificam suavização via Q-values.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.KLDivergence(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
        tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc'),
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc'),
    ],
)

model.summary()

Model: "BoxNet_v3_ArenaSagaz"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 tabuleiro_cru (InputLayer)     [(None, 9, 7, 1)]    0           []                               
                                                                                                  
 grid_de_caixas (Lambda)        (None, 4, 3, 5)      0           ['tabuleiro_cru[0][0]']          
                                                                                                  
 conv2d (Conv2D)                (None, 4, 3, 32)     1440        ['grid_de_caixas[0][0]']         
                                                                                                  
 batch_normalization (BatchNorm  (None, 4, 3, 32)    128         ['conv2d[0][0]']                 
 alization)                                                                    

In [5]:
# EarlyStopping em val_loss (mais sensível ao overfit que val_accuracy).
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1),
]

history = model.fit(
    X_train, y_train,
    epochs=120,
    batch_size=256,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=2,
)

Epoch 1/120
3282/3282 - 37s - loss: 0.7821 - accuracy: 0.2616 - top3_acc: 0.5234 - top5_acc: 0.6466 - val_loss: 0.1890 - val_accuracy: 0.3241 - val_top3_acc: 0.6548 - val_top5_acc: 0.7900 - lr: 0.0010 - 37s/epoch - 11ms/step
Epoch 2/120
3282/3282 - 33s - loss: 0.3279 - accuracy: 0.3512 - top3_acc: 0.6855 - top5_acc: 0.8150 - val_loss: 0.1337 - val_accuracy: 0.3259 - val_top3_acc: 0.6557 - val_top5_acc: 0.8062 - lr: 0.0010 - 33s/epoch - 10ms/step
Epoch 3/120
3282/3282 - 33s - loss: 0.2967 - accuracy: 0.3511 - top3_acc: 0.6866 - top5_acc: 0.8166 - val_loss: 0.1325 - val_accuracy: 0.2914 - val_top3_acc: 0.6026 - val_top5_acc: 0.7678 - lr: 0.0010 - 33s/epoch - 10ms/step
Epoch 4/120
3282/3282 - 33s - loss: 0.2879 - accuracy: 0.3519 - top3_acc: 0.6872 - top5_acc: 0.8174 - val_loss: 0.1401 - val_accuracy: 0.5404 - val_top3_acc: 0.8725 - val_top5_acc: 0.9636 - lr: 0.0010 - 33s/epoch - 10ms/step
Epoch 5/120
3282/3282 - 33s - loss: 0.2842 - accuracy: 0.3515 - top3_acc: 0.6863 - top5_acc: 0.8165 

KeyboardInterrupt: 

In [ ]:
# =========================================================================
# 4. AVALIAÇÃO TEXTUAL — métricas para copiar/colar e comparar entre versões
#
# Com soft targets, y_train/val/test são distribuições de probabilidade.
# - model.evaluate retorna KLD (não cross-entropy) — magnitudes diferentes.
#   O que importa é a tendência (descendo) e o gap treino/val.
# - CategoricalAccuracy e TopKCategoricalAccuracy fazem argmax(y_true)
#   internamente, então top-1/top-3/top-5 continuam interpretáveis como
#   "o modelo escolheu a melhor jogada do Minimax no top-K?".
# =========================================================================

def avalia_conjunto(nome, X, y_soft):
    m = model.evaluate(X, y_soft, verbose=0, return_dict=True)
    return {
        'conjunto': nome,
        'amostras': X.shape[0],
        'kld_loss':  m['loss'],
        'top1_acc':  m['accuracy'],
        'top3_acc':  m['top3_acc'],
        'top5_acc':  m['top5_acc'],
    }

resultados = pd.DataFrame([
    avalia_conjunto('Treino',    X_train, y_train),
    avalia_conjunto('Validação', X_val,   y_val),
    avalia_conjunto('Teste',     X_test,  y_test),
])
print("=" * 70)
print("RESUMO DE DESEMPENHO (BoxNet v3 — soft targets / KL Divergence)")
print("=" * 70)
print(resultados.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

gap_top1 = resultados.loc[0, 'top1_acc'] - resultados.loc[1, 'top1_acc']
gap_kld  = resultados.loc[1, 'kld_loss'] - resultados.loc[0, 'kld_loss']
print(f"\nGap top1 (Treino - Val): {gap_top1*100:+.2f} pp   [< 5 pp saudável; > 10 pp = overfit]")
print(f"Gap KLD  (Val - Treino): {gap_kld:+.4f}")

ult = len(history.history['loss']) - 1
print(f"\nÚltima época: {ult+1}")
print(f"  kld_loss  treino={history.history['loss'][ult]:.4f}  val={history.history['val_loss'][ult]:.4f}")
print(f"  top1_acc  treino={history.history['accuracy'][ult]:.4f}  val={history.history['val_accuracy'][ult]:.4f}")

# =========================================================================
# Classification report — usa argmax do soft target como rótulo verdadeiro
# (argmax do Q-value = melhor jogada do Minimax, mesma semântica de antes)
# =========================================================================
y_pred_prob = model.predict(X_test, verbose=0)
y_pred_idx  = y_pred_prob.argmax(axis=1)

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT (conjunto de TESTE)")
print("=" * 70)
report = classification_report(
    y_test_idx, y_pred_idx,
    labels=list(range(num_classes)),
    target_names=[indice_para_rotulo[i] for i in range(num_classes)],
    digits=4, zero_division=0, output_dict=True,
)
print(f"  accuracy:      {report['accuracy']:.4f}")
print(f"  macro avg:     P={report['macro avg']['precision']:.4f}  "
      f"R={report['macro avg']['recall']:.4f}  F1={report['macro avg']['f1-score']:.4f}")
print(f"  weighted avg:  P={report['weighted avg']['precision']:.4f}  "
      f"R={report['weighted avg']['recall']:.4f}  F1={report['weighted avg']['f1-score']:.4f}")

por_classe = pd.DataFrame({
    rotulo: report[rotulo]
    for rotulo in [indice_para_rotulo[i] for i in range(num_classes)]
}).T
por_classe.index.name = 'jogada'
por_classe = por_classe.sort_values('f1-score', ascending=False)

print("\nTop 10 jogadas com melhor F1 (onde o modelo brilha):")
print(por_classe.head(10)[['precision', 'recall', 'f1-score', 'support']]
      .to_string(float_format=lambda v: f"{v:.4f}"))
print("\nBottom 5 jogadas (onde o modelo mais erra — verificar bordas):")
print(por_classe.tail(5)[['precision', 'recall', 'f1-score', 'support']]
      .to_string(float_format=lambda v: f"{v:.4f}"))

# =========================================================================
# Gráficos de treino
# =========================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].plot(history.history['accuracy'],     label='Treino')
axes[0].plot(history.history['val_accuracy'], label='Validação')
axes[0].set_title('Top-1 Accuracy')
axes[0].set_xlabel('Época'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history.history['top3_acc'],     label='Treino')
axes[1].plot(history.history['val_top3_acc'], label='Validação')
axes[1].set_title('Top-3 Accuracy')
axes[1].set_xlabel('Época'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(history.history['loss'],     label='Treino (KLD)')
axes[2].plot(history.history['val_loss'], label='Validação (KLD)')
axes[2].set_title('KL Divergence Loss')
axes[2].set_xlabel('Época'); axes[2].legend(); axes[2].grid(True)

plt.suptitle('BoxNet v3 — Soft Targets / KL Divergence', fontsize=13)
plt.tight_layout(); plt.show()
# =========================================================================
# 5. METRICAS DIAGNOSTICAS EXTRAS
# =========================================================================

# 5.1  Top-1 / Top-3 por fase do jogo
# Conta tracados preenchidos nas posicoes H (r par, c impar) e V (r impar, c par)
mascara_tracos = np.zeros((9, 7), dtype=bool)
for _r in range(9):
    for _c in range(7):
        if (_r % 2 == 0 and _c % 2 == 1) or (_r % 2 == 1 and _c % 2 == 0):
            mascara_tracos[_r, _c] = True
tracos_jogados = (X_test[:, :, :, 0] != 0)[:, mascara_tracos].sum(axis=1)

fases = [
    (0,  10, 'Abertura  (0-10 tracos)'),
    (11, 20, 'Meio-jogo (11-20 tracos)'),
    (21, 31, 'Final     (21-31 tracos)'),
]
print('\n' + '=' * 70)
print('TOP-1 / TOP-3 ACCURACY POR FASE DO JOGO')
print('=' * 70)
print(f"  {'Fase':<28}  {'N':>6}  {'Top-1':>7}  {'Top-3':>7}")
for lo, hi, nome in fases:
    mask = (tracos_jogados >= lo) & (tracos_jogados <= hi)
    if mask.sum() == 0:
        continue
    t1 = (y_pred_idx[mask] == y_test_idx[mask]).mean()
    top3_pred = np.argsort(y_pred_prob[mask], axis=1)[:, -3:]
    t3 = (top3_pred == y_test_idx[mask, np.newaxis]).any(axis=1).mean()
    print(f"  {nome:<28}  {mask.sum():>6}  {t1:>6.1%}  {t3:>6.1%}")

# 5.2  Optimal Move Accuracy
# A previsao top-1 pertence ao conjunto de jogadas Minimax-otimas?
# S_test contem Q-values brutos; otimo = score == max_score (e slot disponivel)
max_scores_test = S_test.max(axis=1, keepdims=True)
eh_otimo       = (S_test == max_scores_test) & (S_test > -1e8)
pred_eh_otimo  = eh_otimo[np.arange(len(y_pred_idx)), y_pred_idx]
optimal_acc    = pred_eh_otimo.mean()
media_equiv    = eh_otimo.sum(axis=1).mean()
print(f'\n  Optimal Move Accuracy (previsao in conjunto Minimax-otimo): {optimal_acc:.1%}')
print(f'  Media de jogadas Minimax-equivalentes por estado: {media_equiv:.1f}')
print()


In [ ]:
# =========================================================================
# 6. EXPORTAÇÃO PARA TENSORFLOW LITE (Edge AI)
# =========================================================================
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# Otimizações padrão: quantização dinâmica de pesos (~4x menor sem perda significativa).
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

nome_arquivo_tflite = '../modelos/pontinhos_pequeno.tflite'
with open(nome_arquivo_tflite, 'wb') as f:
    f.write(tflite_model)

tamanho_kb = len(tflite_model) / 1024
print(f"Modelo salvo: {nome_arquivo_tflite}  ({tamanho_kb:.1f} KB)")
print(f"Total de parâmetros do Keras: {model.count_params():,}")

In [7]:
# =========================================================================
# 7. AVALIAÇÃO POR PARTIDAS REAIS — CNN vs Minimax
#
# Joga partidas completas entre a CNN e o Minimax em diferentes
# profundidades. Metade das partidas com CNN como primeiro jogador,
# metade como segundo. Mede taxa de vitória e velocidade de decisão.
# =========================================================================
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from gerador_dados.avaliador_partidas import (
    _cnn_agent_fn, _minimax_agent_fn, avaliar, imprimir_relatorio,
    todos_labels_canonicos
)

CAMINHO_MODELO  = '../modelos/pontinhos_pequeno.tflite'
TAMANHO         = 'pequeno'
N_PARTIDAS      = 200    # total por profundidade (100 CNN 1o + 100 CNN 2o)
PROFUNDIDADES   = [1, 3, 5, 6]

labels     = todos_labels_canonicos(4, 3)
agente_cnn = _cnn_agent_fn(CAMINHO_MODELO, labels)

stats_list = []
for prof in PROFUNDIDADES:
    nome = f'Minimax(p={prof})'
    print(f'Avaliando contra {nome}...')
    agente_mm = _minimax_agent_fn(prof)
    s = avaliar(agente_cnn, agente_mm, nome, TAMANHO, N_PARTIDAS)
    stats_list.append(s)

imprimir_relatorio(stats_list)


ValueError: Didn't find op for builtin opcode 'FULLY_CONNECTED' version '12'. An older version of this builtin might be supported. Are you using an old TFLite binary with a newer model?
Registration failed.
